___

___

# Latent Dirichlet Allocation (LDA)

## Datos

Usaremos artículos de Radio Pública Nacional, obtenidos de su sitio web [www.npr.org](http://www.npr.org)

In [1]:
import pandas as pd

In [6]:
npr = pd.read_csv('../datos/npr.csv')

In [7]:
npr.head()

,Article
0,"In the Washington of 2016, even when the polic..."
1,Donald Trump has used Twitter — his prefe...
2,Donald Trump is unabashedly praising Russian...
3,"Updated at 2:50 p. m. ET, Russian President Vl..."
4,"From photography, illustration and video, to d..."


¡Observe cómo no tenemos un campo que indique el tema del que hablan en los artículos! Usemos LDA para intentar descifrar grupos de artículos.

## Preprocesamiento

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

/opt/anaconda3/envs/curso_nlp/lib/python3.7/site-packages/sklearn/utils/validation.py:37: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  LARGE_SPARSE_SUPPORTED = LooseVersion(scipy_version) >= '0.14.0'


**`max_df`**` : flotante en rango [0.0, 1.0] o int, predeterminado=1.0`<br>
Al construir el vocabulario, ignora los términos que tienen una frecuencia de documento estrictamente superior al umbral dado (palabras vacías específicas del corpus). Si es flotante, el parámetro representa una proporción de documentos, números absolutos enteros. Este parámetro se ignora si el vocabulario no es None.

**`min_df`**` : flotante en rango [0.0, 1.0] o int, predeterminado=1`<br>
Al construir el vocabulario, ignora los términos que tienen una frecuencia de documento estrictamente inferior al umbral dado. Este valor también se denomina corte en la literatura. Si es flotante, el parámetro representa una proporción de documentos, números absolutos enteros. Este parámetro se ignora si el vocabulario no es None.

In [10]:
cv = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')

In [11]:
dtm = cv.fit_transform(npr['Article'])

In [12]:
dtm

<11992x54777 sparse matrix of type '<class 'numpy.int64'>'
	with 3033388 stored elements in Compressed Sparse Row format>

## LDA

In [13]:
from sklearn.decomposition import LatentDirichletAllocation

In [14]:
LDA = LatentDirichletAllocation(n_components=7,random_state=42)

In [15]:
# ¡Esto puede llevar un tiempo, estamos tratando con una gran cantidad de documentos!
LDA.fit(dtm)

LatentDirichletAllocation(batch_size=128, doc_topic_prior=None,
             evaluate_every=-1, learning_decay=0.7,
             learning_method='batch', learning_offset=10.0,
             max_doc_update_iter=100, max_iter=10, mean_change_tol=0.001,
             n_components=7, n_jobs=None, n_topics=None, perp_tol=0.1,
             random_state=42, topic_word_prior=None,
             total_samples=1000000.0, verbose=0)

## Mostrando palabras almacenadas

In [16]:
len(cv.get_feature_names())

54777

In [17]:
import random

In [18]:
for i in range(10):
    random_word_id = random.randint(0,54776)
    print(cv.get_feature_names()[random_word_id])

curatorial
sousley
streeter
airplane
sumac
sandra
anesthesia
tipped
antithetical
troy


In [19]:
# Generemos otros 10 textos
for i in range(10):
    random_word_id = random.randint(0,54776)
    print(cv.get_feature_names()[random_word_id])

wto
cox
airfare
protestant
referring
vomiting
sap
surveyed
sandtown
bluefish


### Mostrando las palabras principales por tema

In [20]:
len(LDA.components_)

7

In [21]:
LDA.components_

array([[8.64332806e+00, 2.38014333e+03, 1.42900522e-01, ...,
        1.43006821e-01, 1.42902042e-01, 1.42861626e-01],
       [2.76191749e+01, 5.36394437e+02, 1.42857148e-01, ...,
        1.42861973e-01, 1.42857147e-01, 1.42906875e-01],
       [7.22783888e+00, 8.24033986e+02, 1.42857148e-01, ...,
        6.14236247e+00, 2.14061364e+00, 1.42923753e-01],
       ...,
       [3.11488651e+00, 3.50409655e+02, 1.42857147e-01, ...,
        1.42859912e-01, 1.42857146e-01, 1.42866614e-01],
       [4.61486388e+01, 5.14408600e+01, 3.14281373e+00, ...,
        1.43107628e-01, 1.43902481e-01, 2.14271779e+00],
       [4.93991422e-01, 4.18841042e+02, 1.42857151e-01, ...,
        1.42857146e-01, 1.43760101e-01, 1.42866201e-01]])

In [22]:
len(LDA.components_[0])

54777

In [23]:
single_topic = LDA.components_[0]

In [24]:
# Devuelve los índices que ordenarían esta matriz.
single_topic.argsort()

array([ 2475, 18302, 35285, ..., 22673, 42561, 42993])

In [25]:
# Palabra menos representativa de este tema
single_topic[2475]

0.1428571430851871

In [26]:
# Word most representative of this topic
single_topic[42993]

6247.245510521018

In [27]:
# Las 10 palabras principales para este tema:
single_topic.argsort()[-10:]

array([33390, 36310, 21228, 10425, 31464,  8149, 36283, 22673, 42561,
       42993])

In [29]:
top_word_indices = single_topic.argsort()[-10:]

In [30]:
for index in top_word_indices:
    print(cv.get_feature_names()[index])

new
percent
government
company
million
care
people
health
said
says


Quizás parezcan artículos comerciales... Confirmemos usando .transform() en nuestros artículos vectorizados para adjuntar un número de etiqueta. Pero primero, veamos los 10 temas encontrados.

In [34]:
for index,topic in enumerate(LDA.components_):
    print(f'LAS 10 PALABRAS PRINCIPALES PARA EL TEMA #{index}')
    print([cv.get_feature_names()[i] for i in topic.argsort()[-10:]])
    print('\n')

LAS 10 PALABRAS PRINCIPALES PARA EL TEMA #0
['new', 'percent', 'government', 'company', 'million', 'care', 'people', 'health', 'said', 'says']


LAS 10 PALABRAS PRINCIPALES PARA EL TEMA #1
['npr', 'reports', 'says', 'news', 'people', 'told', 'police', 'president', 'trump', 'said']


LAS 10 PALABRAS PRINCIPALES PARA EL TEMA #2
['time', 'water', 'city', 'new', 'years', 'food', 'just', 'people', 'like', 'says']


LAS 10 PALABRAS PRINCIPALES PARA EL TEMA #3
['disease', 'patients', 'just', 'children', 'study', 'like', 'women', 'health', 'people', 'says']


LAS 10 PALABRAS PRINCIPALES PARA EL TEMA #4
['obama', 'court', 'republican', 'campaign', 'people', 'state', 'president', 'clinton', 'said', 'trump']


LAS 10 PALABRAS PRINCIPALES PARA EL TEMA #5
['new', 'way', 'music', 'really', 'time', 'know', 'think', 'people', 'just', 'like']


LAS 10 PALABRAS PRINCIPALES PARA EL TEMA #6
['people', 'time', 'schools', 'just', 'education', 'new', 'like', 'students', 'school', 'says']




### Adjuntar etiquetas de temas descubiertos a artículos originales

In [35]:
dtm

<11992x54777 sparse matrix of type '<class 'numpy.int64'>'
	with 3033388 stored elements in Compressed Sparse Row format>

In [36]:
dtm.shape

(11992, 54777)

In [37]:
len(npr)

11992

In [38]:
topic_results = LDA.transform(dtm)

In [39]:
topic_results.shape

(11992, 7)

In [40]:
topic_results[0]

array([1.61040465e-02, 6.83341493e-01, 2.25376318e-04, 2.25369288e-04,
       2.99652737e-01, 2.25479379e-04, 2.25497980e-04])

In [41]:
topic_results[0].round(2)

array([0.02, 0.68, 0.  , 0.  , 0.3 , 0.  , 0.  ])

In [42]:
topic_results[0].argmax()

1

Esto significa que nuestro modelo piensa que el primer artículo pertenece al tema #1.

### Combinación con datos originales

In [44]:
npr.head()

,Article
0,"In the Washington of 2016, even when the polic..."
1,Donald Trump has used Twitter — his prefe...
2,Donald Trump is unabashedly praising Russian...
3,"Updated at 2:50 p. m. ET, Russian President Vl..."
4,"From photography, illustration and video, to d..."


In [45]:
topic_results.argmax(axis=1)

array([1, 1, 1, ..., 3, 4, 0])

In [46]:
npr['Topic'] = topic_results.argmax(axis=1)

In [47]:
npr.head(10)

,Article,Topic
0,"In the Washington of 2016, even when the polic...",1
1,Donald Trump has used Twitter — his prefe...,1
2,Donald Trump is unabashedly praising Russian...,1
3,"Updated at 2:50 p. m. ET, Russian President Vl...",1
4,"From photography, illustration and video, to d...",2
5,I did not want to join yoga class. I hated tho...,3
6,With a who has publicly supported the debunk...,3
7,"I was standing by the airport exit, debating w...",2
8,"If movies were trying to be more realistic, pe...",3
9,"Eighteen years ago, on New Year’s Eve, David F...",2


## Excelente trabajo!